# MuIS andmete töötlemine

## 🚀 Kui jätkad poolelijäänud tööst:
1. Muuda `START_FROM_BATCH = 2` (batch nr, kust jätkata)
2. Käivita kõik lahtrid järjekorras
3. Liitmise lahtris lae üles ka varasemad batch failid

In [ ]:
!pip install anthropic openpyxl --break-system-packages -q

import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re
from tqdm import tqdm
import time
import json
from anthropic import Anthropic
from datetime import datetime

print('✓ Teegid installitud')

✓ Teegid installitud


In [ ]:
# ============================================================
# SEADISTUSED
# ============================================================

CLAUDE_API_KEY = 'buy_your_own'
BATCH_SIZE = 500                # Batch suurus

TEST_LIMIT = None               # None = kõik objektid
                                # 20   = testimiseks 20 objekti

START_FROM_BATCH = 17            # 1 = algusest
                                # 2 = jätka batch 2-st (batch 1 on juba valmis)
                                # 3 = jätka batch 3-st jne

# ============================================================

if TEST_LIMIT and TEST_LIMIT <= 100:
    BATCH_SIZE = TEST_LIMIT
    START_FROM_BATCH = 1
    print(f'⚠️  TESTIMINE: {TEST_LIMIT} objekti ühes batch\'is')

claude_client = Anthropic(api_key=CLAUDE_API_KEY)
print(f'✓ Batch suurus: {BATCH_SIZE}')
print(f'✓ Alustan batch\'ist nr: {START_FROM_BATCH}')
print(f'✓ Test limit: {TEST_LIMIT or "Kõik"}')

✓ Batch suurus: 500
✓ Alustan batch'ist nr: 17
✓ Test limit: Kõik


In [ ]:
from google.colab import files
uploaded = files.upload()
collection_rdf_path = list(uploaded.keys())[0]
print(f'✓ Fail: {collection_rdf_path}')

Saving resource-2.rdf to resource-2.rdf
✓ Fail: resource-2.rdf


## 🔧 Abifunktsioonid (safe_join, fetch_rdf, thesaurus, events)

In [ ]:
def safe_join(value):
    """Turvaline list->string. Käsitleb kõiki ootamatuid Claude vastuse formaate."""
    if not value:
        return None
    if isinstance(value, str):
        return value.strip() or None
    if isinstance(value, list):
        flat = []
        for item in value:
            if isinstance(item, list):
                flat.extend([str(i).strip() for i in item if i])
            elif isinstance(item, dict):
                flat.extend([str(v).strip() for v in item.values() if v])
            elif item:
                flat.append(str(item).strip())
        result = ', '.join(f for f in flat if f)
        return result or None
    return str(value) if value else None


def extract_object_ids(rdf_file_path):
    tree = ET.parse(rdf_file_path)
    root = tree.getroot()
    ns = {'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
          'crm': 'http://www.cidoc-crm.org/cidoc-crm/'}
    return [e.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
            for e in root.findall('.//crm:P46_is_composed_of', ns)
            if e.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')]


def fetch_rdf(url, max_retries=3):
    headers = {'Accept': 'application/rdf+xml'}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                return r.content
            elif r.status_code == 404:
                return None
        except requests.exceptions.RequestException:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None


thesaurus_cache = {}

def fetch_thesaurus_label(url):
    if not url:
        return None
    if url in thesaurus_cache:
        return thesaurus_cache[url]
    rdf = fetch_rdf(url)
    if rdf is None:
        thesaurus_cache[url] = None
        return None
    try:
        root = ET.fromstring(rdf)
        ns = {'rdfs': 'http://www.w3.org/2000/01/rdf-schema#'}
        lbl = root.find('.//rdfs:label', ns)
        result = lbl.text.strip() if lbl is not None and lbl.text else None
        thesaurus_cache[url] = result
        return result
    except ET.ParseError:
        thesaurus_cache[url] = None
        return None


event_cache = {}
event_log = {'success': 0, 'failed': [], 'total': 0}

def fetch_event_name(event_url):
    if not event_url:
        return None
    if event_url in event_cache:
        return event_cache[event_url]
    event_log['total'] += 1
    rdf = fetch_rdf(event_url)
    if rdf is None:
        event_log['failed'].append(event_url.split('/')[-1])
        event_cache[event_url] = None
        return None
    try:
        root = ET.fromstring(rdf)
        ns = {'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
              'crm':  'http://www.cidoc-crm.org/cidoc-crm/',
              'dcterms': 'http://purl.org/dc/terms/'}
        for tag in ['.//rdfs:label', './/dcterms:title']:
            elem = root.find(tag, ns)
            if elem is not None and elem.text:
                name = elem.text.strip()
                event_cache[event_url] = name
                event_log['success'] += 1
                return name
        te = root.find('.//crm:P2_has_type', ns)
        if te is not None:
            tu = te.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
            if tu:
                lbl = fetch_thesaurus_label(tu)
                if lbl:
                    event_cache[event_url] = lbl
                    event_log['success'] += 1
                    return lbl
        event_log['failed'].append(event_url.split('/')[-1])
        event_cache[event_url] = None
        return None
    except ET.ParseError:
        event_log['failed'].append(event_url.split('/')[-1])
        event_cache[event_url] = None
        return None


print('✓ Abifunktsioonid OK (safe_join, fetch_rdf, thesaurus, events)')

✓ Abifunktsioonid OK (safe_join, fetch_rdf, thesaurus, events)


## 📅 Kuupäeva parsimine (rooma numbrid → ISO)

In [ ]:
ROMAN_MONTHS = {
    'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6,
    'VII':7,'VIII':8,'IX':9,'X':10,'XI':11,'XII':12
}
ESTONIAN_MONTHS = {
    'jaanuar':1,'veebruar':2,'märts':3,'aprill':4,
    'mai':5,'juuni':6,'juuli':7,'august':8,
    'september':9,'oktoober':10,'november':11,'detsember':12,
    'jaanuaris':1,'veebruaris':2,'märtsis':3,'aprillis':4,
    'mais':5,'juunis':6,'juulis':7,'augustis':8,
    'septembris':9,'oktoobris':10,'novembris':11,'detsembris':12,
    'jaanuarini':1,'veebruarini':2,'märtsini':3,'aprillini':4,
    'maini':5,'juunini':6,'juulini':7,'augustini':8,
    'septembrini':9,'oktoobrini':10,'novembrini':11,'detsembrini':12
}

def roman_to_month(s): return ROMAN_MONTHS.get(s.upper())
def estonian_to_month(s): return ESTONIAN_MONTHS.get(s.lower())

def format_date(day, month, year):
    try:
        if year and month and day:
            return f'{int(year):04d}-{int(month):02d}-{int(day):02d}'
        elif year and month:
            return f'{int(year):04d}-{int(month):02d}'
        elif year:
            return str(int(year))
    except: pass
    return None

def parse_collection_date(text):
    if not text: return {'start': None, 'end': None}
    t = str(text).strip()
    RM = r'(I{1,3}|I?V|VI{0,3}|IX|XI{0,2}|XII)'

    # DD.RM.-DD.RM.YYYY  (vahemik, eri kuud)
    m = re.search(rf'(\d{{1,2}})\.?{RM}\.?[-–](\d{{1,2}})\.?{RM}\.?(\d{{4}})', t, re.I)
    if m:
        d1,rm1,d2,rm2,y = m.group(1),m.group(2),m.group(3),m.group(4),m.group(5)
        mo1,mo2 = roman_to_month(rm1), roman_to_month(rm2)
        if mo1 and mo2:
            return {'start': format_date(d1,mo1,y), 'end': format_date(d2,mo2,y)}

    # DD.-DD.RM.YYYY  (vahemik, sama kuu)
    m = re.search(rf'(\d{{1,2}})\.[-–](\d{{1,2}})\.?{RM}\.?(\d{{4}})', t, re.I)
    if m:
        d1,d2,rm,y = m.group(1),m.group(2),m.group(3),m.group(4)
        mo = roman_to_month(rm)
        if mo: return {'start': format_date(d1,mo,y), 'end': format_date(d2,mo,y)}

    # DD.RM.YYYY  (üks kuupäev)
    m = re.search(rf'(\d{{1,2}})\.{RM}\.(\d{{4}})', t, re.I)
    if m:
        d,rm,y = m.group(1),m.group(2),m.group(3)
        mo = roman_to_month(rm)
        if mo:
            dt = format_date(d,mo,y)
            return {'start': dt, 'end': dt}

    # DD.-DD. EESTI_KUU YYYY
    m = re.search(r'(\d{1,2})\.[-–](\d{1,2})\.?\s*([a-zäöüõ]+)\s+(\d{4})', t, re.I)
    if m:
        d1,d2,mn,y = m.groups()
        mo = estonian_to_month(mn)
        if mo: return {'start': format_date(d1,mo,y), 'end': format_date(d2,mo,y)}

    # EESTI_KUU YYYY
    m = re.search(r'([a-zäöüõ]{4,})\s+(\d{4})', t, re.I)
    if m:
        mn,y = m.groups()
        mo = estonian_to_month(mn)
        if mo:
            dt = format_date(None,mo,y)
            return {'start': dt, 'end': dt}

    # YYYY-YYYY
    m = re.search(r'(\d{4})[-–](\d{4})', t)
    if m: return {'start': m.group(1), 'end': m.group(2)}

    # YYYY. ja YYYY.
    m = re.search(r'(\d{4}).*?ja.*?(\d{4})', t)
    if m: return {'start': m.group(1), 'end': m.group(2)}

    # Ainult aasta
    m = re.search(r'(\d{4})', t)
    if m: return {'start': m.group(1), 'end': m.group(1)}

    return {'start': None, 'end': None}


# Testid
tests = [
    ('10.-22.VII.1983.a.',       '1983-07-10','1983-07-22'),
    ('19.IX.-10.X.1989.a.',      '1989-09-19','1989-10-10'),
    ('25.VII.-11.VIII.1973.a.',  '1973-07-25','1973-08-11'),
    ('3.IV.1979.a.',             '1979-04-03','1979-04-03'),
    ('2.-20.juulini 1974',       '1974-07-02','1974-07-20'),
    ('6.-27.VII.1981.a.',        '1981-07-06','1981-07-27'),
    ('1914-1915',                '1914','1915'),
    ('1914. ja 1915.a.',         '1914','1915'),
    ('oktoobris 1925',           '1925-10','1925-10'),
]
print('KUUPÄEVA TESTID:')
all_ok = True
for text, es, ee in tests:
    r = parse_collection_date(text)
    ok = r['start']==es and r['end']==ee
    if not ok: all_ok = False
    print(f'{"✅" if ok else "❌"} {text!r}')
    if not ok: print(f'   start: {r["start"]!r} (oodatud {es!r}), end: {r["end"]!r} (oodatud {ee!r})')
print('✅ KÕIK TESTID OK!' if all_ok else '⚠️  Mõned testid ebaõnnestusid!')

KUUPÄEVA TESTID:
✅ '10.-22.VII.1983.a.'
✅ '19.IX.-10.X.1989.a.'
✅ '25.VII.-11.VIII.1973.a.'
✅ '3.IV.1979.a.'
✅ '2.-20.juulini 1974'
✅ '6.-27.VII.1981.a.'
✅ '1914-1915'
✅ '1914. ja 1915.a.'
✅ 'oktoobris 1925'
✅ KÕIK TESTID OK!


## 🎯 Põhiandmete ekstraheerimine

In [ ]:
def extract_basic_data_from_rdf(rdf_content):
    empty = {'title':None,'museaali_number':None,'image_url':None,
             'material':None,'event_ids':None,'event_names':None,
             'related_objects':None,'relation_type':None}
    if rdf_content is None: return empty
    try:
        root = ET.fromstring(rdf_content)
        ns = {
            'rdf':'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
            'crm':'http://www.cidoc-crm.org/cidoc-crm/',
            'dcterms':'http://purl.org/dc/terms/',
            'rdfs':'http://www.w3.org/2000/01/rdf-schema#',
            'foaf':'http://xmlns.com/foaf/0.1/',
            'owl':'http://www.w3.org/2002/07/owl#'
        }
        r = dict(empty)

        # Title
        for ot in ['crm:E18_Physical_Thing','crm:E22_Man-Made_Object']:
            obj = root.find(f'.//{ot}', ns)
            if obj is not None:
                lbl = obj.find('rdfs:label', ns)
                if lbl is not None and lbl.text:
                    r['title'] = lbl.text.strip(); break

        # Museaali number
        idf = root.find('.//dcterms:identifier', ns)
        if idf is not None and idf.text: r['museaali_number'] = idf.text.strip()

        # Image
        dep = root.find('.//foaf:depiction', ns)
        if dep is not None:
            r['image_url'] = dep.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')

        # Material
        materials = []
        for p45 in root.findall('.//crm:P45_consists_of', ns):
            mat = p45.find('.//crm:E57_Material', ns)
            if mat is not None:
                p130 = mat.find('.//crm:P130_shows_features_of', ns)
                if p130 is not None:
                    mu = p130.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
                    if mu:
                        lbl = fetch_thesaurus_label(mu)
                        if lbl: materials.append(lbl)
                note = mat.find('.//crm:P3_has_note', ns)
                if note is not None and note.text: materials.append(note.text.strip())
        if materials: r['material'] = ', '.join(sorted(set(materials)))

        # Events
        eids, enames = [], []
        for p12 in root.findall('.//crm:P12_occurred_in_the_presence_of', ns):
            eu = p12.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
            if eu:
                eids.append(eu.split('/')[-1])
                name = fetch_event_name(eu)
                if name: enames.append(name)
        if eids: r['event_ids'] = ', '.join(eids)
        if enames: r['event_names'] = ' | '.join(enames)

        # Seotud objektid
        related, rtypes = [], []
        for p130 in root.findall('.//crm:P130_shows_features_of', ns):
            nested = p130.find('.//crm:E18_Physical_Thing', ns)
            if nested is not None:
                sa = nested.find('.//owl:sameAs', ns)
                if sa is not None:
                    ru = sa.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
                    if ru: related.append(ru.split('/')[-1])
                te = nested.find('.//crm:P2_has_type', ns)
                if te is not None:
                    tu = te.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
                    if tu:
                        lbl = fetch_thesaurus_label(tu)
                        if lbl and lbl not in rtypes: rtypes.append(lbl)
        if related: r['related_objects'] = ', '.join(related)
        if rtypes: r['relation_type'] = ', '.join(rtypes)
        return r
    except ET.ParseError: return empty


def create_title_clean(title_rdf):
    if not title_rdf or pd.isna(title_rdf): return None
    t = title_rdf.lower().strip()
    if len(t.split()) > 3: return None
    if re.search(r'\d{4}', title_rdf): return None
    if any(w in t for w in ['kogu','kogutud','ekspeditsioon']): return None
    return title_rdf

print('✓ Põhiandmete funktsioonid OK')

✓ Põhiandmete funktsioonid OK


## 🎯 Descriptions + Claude API

In [ ]:
def extract_descriptions_from_object(rdf_content):
    if rdf_content is None: return []
    try:
        root = ET.fromstring(rdf_content)
        ns = {'rdf':'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
              'crm':'http://www.cidoc-crm.org/cidoc-crm/',
              'rdfs':'http://www.w3.org/2000/01/rdf-schema#'}
        descriptions = []
        for p67 in root.findall('.//crm:P67i_is_referred_to_by', ns):
            ling = p67.find('.//crm:E33_Linguistic_Object', ns)
            if ling is not None:
                te = ling.find('crm:P2_has_type', ns)
                type_url = te.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource') if te is not None else None
                le = ling.find('rdfs:label', ns)
                text = le.text.strip() if le is not None and le.text else None
                if text: descriptions.append({'type_url': type_url, 'text': text})
        return descriptions
    except ET.ParseError: return []


def extract_structured_data_with_claude(description_text):
    prompt = f"""Palun ekstrakteeri järgmisest muuseumikogu kirjeldusest struktureeritud andmed.

KIRJELDUS:
{description_text}

Ekstrakteeri JSON formaadis:
1. "places" - kohanimede list (ainult stringid, nt ["Vologda oblast", "Babajevo rajoon"])
2. "people_names" - rahvuste nimed (ainult stringid, nt ["vepsa"])
3. "collection_date" - kogumise aeg algsel kujul (string, nt "10.-22.VII.1983.a.")
4. "collectors" - kogujate nimed (ainult stringid, nt ["L. Lepp", "A. Peterson"])

OLULINE: listid peavad sisaldama AINULT stringe, mitte listide listisid!
Tagasta AINULT JSON, ilma backtick'eta."""

    empty = {'places':[], 'people_names':[], 'collection_date':None, 'collectors':[]}
    try:
        msg = claude_client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=1000,
            messages=[{'role':'user','content':prompt}]
        )
        txt = msg.content[0].text.strip().replace('```json','').replace('```','').strip()
        data = json.loads(txt)
        for k in ['places','people_names','collectors']:
            if k not in data: data[k] = []
        if 'collection_date' not in data: data['collection_date'] = None
        return data
    except: return empty

print('✓ Descriptions + Claude OK')

✓ Descriptions + Claude OK


## 🚀 Batch töötlemine

In [ ]:
def process_batch(object_urls, batch_number, total_batches):
    print(f'\n{"="*80}\nBATCH {batch_number}/{total_batches} - {len(object_urls)} objekti\n{"="*80}')

    results = []
    unique_type_urls = set()

    print('\nETAPP 1/3: Põhiandmed...')
    for url in tqdm(object_urls, desc='Objektid'):
        object_id = url.split('/')[-1]
        rdf = fetch_rdf(url)
        basic = extract_basic_data_from_rdf(rdf)
        title_clean = create_title_clean(basic['title'])
        descriptions = extract_descriptions_from_object(rdf)
        base = {
            'object_id': object_id, 'object_url': url,
            'museaali_number': basic['museaali_number'], 'image_url': basic['image_url'],
            'material': basic['material'], 'event_ids': basic['event_ids'],
            'event_names': basic['event_names'], 'related_objects': basic['related_objects'],
            'relation_type': basic['relation_type'], 'title_rdf': basic['title'],
            'title_clean': title_clean
        }
        if not descriptions:
            results.append({**base, 'description_type_url':None,
                            'description_type_label':None, 'description_text':None})
        else:
            for desc in descriptions:
                results.append({**base, 'description_type_url':desc['type_url'],
                                'description_type_label':None, 'description_text':desc['text']})
                if desc['type_url']: unique_type_urls.add(desc['type_url'])
        time.sleep(0.1)

    df = pd.DataFrame(results)

    print(f'\nETAPP 2/3: Tüübid ({len(unique_type_urls)})...')
    for type_url in tqdm(unique_type_urls, desc='Tüübid'):
        lbl = fetch_thesaurus_label(type_url)
        df.loc[df['description_type_url'] == type_url, 'description_type_label'] = lbl
        time.sleep(0.1)

    print('\nETAPP 3/3: Claude API...')
    for col in ['places','people_names','collection_date_raw',
                'collection_date_start','collection_date_end','collectors']:
        df[col] = None

    komm = df[df['description_type_label'] == 'kommentaar']
    if len(komm) > 0:
        print(f'Töötlen {len(komm)} kommentaari...')
        for idx in tqdm(komm.index, desc='Claude API'):
            desc_text = df.loc[idx, 'description_text']
            if desc_text:
                try:
                    ext = extract_structured_data_with_claude(desc_text)
                    df.loc[idx, 'places']       = safe_join(ext.get('places'))
                    df.loc[idx, 'people_names'] = safe_join(ext.get('people_names'))
                    df.loc[idx, 'collectors']   = safe_join(ext.get('collectors'))
                    raw = ext.get('collection_date')
                    if raw and not isinstance(raw, str): raw = str(raw)
                    df.loc[idx, 'collection_date_raw'] = raw
                    if raw:
                        parsed = parse_collection_date(raw)
                        df.loc[idx, 'collection_date_start'] = parsed['start']
                        df.loc[idx, 'collection_date_end']   = parsed['end']
                except Exception as e:
                    print(f'\n⚠️  Objekt {df.loc[idx,"object_id"]} viga: {e} - jätkan...')
            time.sleep(0.5)
    return df

print('✓ Batch funktsioon OK')

✓ Batch funktsioon OK


## 📋 Lae objektid + jaga batch'ideks

In [ ]:
all_object_urls = extract_object_ids(collection_rdf_path)
if TEST_LIMIT:
    all_object_urls = all_object_urls[:TEST_LIMIT]
    print(f'⚠️  TEST: {len(all_object_urls)} objekti')
else:
    print(f'Leitud {len(all_object_urls)} objekti')

all_batches = [all_object_urls[i:i+BATCH_SIZE]
               for i in range(0, len(all_object_urls), BATCH_SIZE)]
total_batches = len(all_batches)
start_idx = START_FROM_BATCH - 1
batches_to_process = all_batches[start_idx:]

print(f'Batch\'e kokku: {total_batches}')
print(f'Töötlen batch\'id: {START_FROM_BATCH} → {total_batches}')
if START_FROM_BATCH > 1:
    missing = [f'muis_batch_{i:03d}_of_{total_batches:03d}.xlsx' for i in range(1, START_FROM_BATCH)]
    print(f'\n💡 Liitmisel lae üles ka: {", ".join(missing)}')

Leitud 10198 objekti
Batch'e kokku: 21
Töötlen batch'id: 17 → 21

💡 Liitmisel lae üles ka: muis_batch_001_of_021.xlsx, muis_batch_002_of_021.xlsx, muis_batch_003_of_021.xlsx, muis_batch_004_of_021.xlsx, muis_batch_005_of_021.xlsx, muis_batch_006_of_021.xlsx, muis_batch_007_of_021.xlsx, muis_batch_008_of_021.xlsx, muis_batch_009_of_021.xlsx, muis_batch_010_of_021.xlsx, muis_batch_011_of_021.xlsx, muis_batch_012_of_021.xlsx, muis_batch_013_of_021.xlsx, muis_batch_014_of_021.xlsx, muis_batch_015_of_021.xlsx, muis_batch_016_of_021.xlsx


## ▶️ KÄIVITA

In [ ]:
all_batch_dfs = []
batch_files = []
start_time = datetime.now()

for i, batch in enumerate(batches_to_process, START_FROM_BATCH):
    df_batch = process_batch(batch, i, total_batches)
    fname = f'muis_batch_{i:03d}_of_{total_batches:03d}.xlsx'

    # Salvesta lokaalselt
    df_batch.to_excel(fname, index=False, engine='openpyxl')

    # ⭐ Salvesta kohe Drive'i
    import shutil
    shutil.copy(fname, f'/content/drive/MyDrive/MUIS/{fname}')

    all_batch_dfs.append(df_batch)
    batch_files.append(fname)
    print(f'\n✓ Batch {i}/{total_batches} VALMIS → {fname}')
    print(f'💾 Salvestatud Drive/MUIS/{fname}')

duration = datetime.now() - start_time
print(f'\n✅ KÕIK VALMIS! Aeg: {duration}')


BATCH 17/21 - 500 objekti

ETAPP 1/3: Põhiandmed...


Objektid: 100%|██████████| 500/500 [1:10:02<00:00,  8.40s/it]



ETAPP 2/3: Tüübid (7)...


Tüübid: 100%|██████████| 7/7 [00:08<00:00,  1.21s/it]



ETAPP 3/3: Claude API...
Töötlen 248 kommentaari...


...


ETAPP 2/3: Tüübid (5)...


Tüübid: 100%|██████████| 5/5 [00:00<00:00,  9.78it/s]



ETAPP 3/3: Claude API...
Töötlen 103 kommentaari...


Claude API: 100%|██████████| 103/103 [04:29<00:00,  2.61s/it]


✓ Batch 21/21 VALMIS → muis_batch_021_of_021.xlsx
💾 Salvestatud Drive/MUIS/muis_batch_021_of_021.xlsx

✅ KÕIK VALMIS! Aeg: 5:40:26.368765


In [ ]:
# See käivitub kohe kui hetkel töötav batch lõpeb
from google.colab import drive
drive.mount('/content/drive')
import shutil, glob, os

os.makedirs('/content/drive/MyDrive/MUIS', exist_ok=True)
for f in sorted(glob.glob('muis_batch_*.xlsx')):
    shutil.copy(f, f'/content/drive/MyDrive/MUIS/{f}')
    print(f'✓ {f}')

Mounted at /content/drive


## 🔗 LIITMINE + SALVESTAMINE

In [ ]:
# Kui alustasid batch 2-st, lae üles ka varasemad batch failid
if START_FROM_BATCH > 1:
    print(f'⚠️  Lae üles batch failid 1 kuni {START_FROM_BATCH - 1}:')
    uploaded_batches = files.upload()
    earlier_dfs = []
    for fname in sorted(uploaded_batches.keys()):
        df_e = pd.read_excel(fname)
        earlier_dfs.append(df_e)
        print(f'  ✓ {fname} ({len(df_e)} rida)')
    df_final = pd.concat(earlier_dfs + all_batch_dfs, ignore_index=True)
else:
    df_final = pd.concat(all_batch_dfs, ignore_index=True)

cols = ['object_id','object_url','museaali_number','image_url','material',
        'event_ids','event_names','related_objects','relation_type',
        'title_rdf','title_clean','description_type_label','description_text',
        'places','people_names','collection_date_raw',
        'collection_date_start','collection_date_end','collectors']
df_export = df_final[[c for c in cols if c in df_final.columns]]

fname = f'muis_FINAL_v5_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
df_export.to_excel(fname, index=False, engine='openpyxl')
print(f'\n✅ LÕPLIK FAIL: {fname}')
print(f'Ridu: {len(df_export)}')
print(f'Objekte: {df_export["object_id"].nunique()}')
files.download(fname)

## 📊 STATISTIKA + COVERAGE CHECK

In [ ]:
print('='*80)
actual = df_export['object_id'].nunique()
expected = len(all_object_urls)
print(f'COVERAGE: {actual}/{expected} objekti')
print('✅ Kõik töödeldud!' if actual == expected else f'⚠️  PUUDUB {expected-actual} objekti!')

komm = df_export[df_export['description_type_label'] == 'kommentaar']
print(f'\nKUUPÄEVAD (kommentaaridest {len(komm)} rida):')
print(f'  raw: {komm["collection_date_raw"].notna().sum()}')
print(f'  start parsitud: {komm["collection_date_start"].notna().sum()}')
print(f'  end parsitud: {komm["collection_date_end"].notna().sum()}')

print(f'\nEVENT LOGI:')
print(f'  Päringuid: {event_log["total"]}, edukaid: {event_log["success"]}, ebaõnnestunuid: {len(event_log["failed"])}')

print(f'\nMATERJALID: {df_export["material"].notna().sum()}')
print(f'SEOTUD OBJEKTID: {df_export["related_objects"].notna().sum()}')
print(f'\nAEG: {duration}')